In [26]:

!pip install -q sentence-transformers chromadb rank-bm25



In [27]:

from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/Colab Notebooks/AI_Recruitment"


!pip install -q deltalake
from deltalake import DeltaTable

silver = DeltaTable(f"{BASE}/delta/silver_applications").to_pandas()
jobs   = pd.read_csv(f"{BASE}/data/jobs.csv")

print("Orders:", len(silver), "| Jobs:", len(jobs))
silver.head(3)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Orders: 33 | Jobs: 5


,job_id,applicant_id,name,education,applicant_experience,skills,title,company,city,salary,required_skills,required_experience,application_id
0,1,1001,Updated B,PhD,11,"Vue.js, JavaScript",Frontend Developer,Tech Vision,Riyadh,12000,"Vue.js,JavaScript,HTML,CSS",2,1001_1
1,1,1001,Updated B,PhD,11,"Vue.js, JavaScript",Frontend Developer,Tech Vision,Riyadh,12000,"Vue.js,JavaScript,HTML,CSS",2,1001_1
2,1,1001,Updated B,PhD,11,"Vue.js, JavaScript",Frontend Developer,Tech Vision,Riyadh,12000,"Vue.js,JavaScript,HTML,CSS",2,1001_1


In [28]:
docs = []


for _, j in jobs.iterrows():
    docs.append({
        "id": f"job_{j['job_id']}",
        "type": "job",
        "source": f"Job #{j['job_id']} — {j['title']}",
        "text": (
            f"Job: {j['title']}  in company {j['company']} "
            f"city {j['city']}. salary: {j['salary']} riyal. "
            f" requied skilles: {j['required_skills']}. "
            f" experience years: {j['experience_years']} years."
        )
    })


for _, a in silver.iterrows():
    docs.append({
        "id": f"app_{a['applicant_id']}",
        "type": "applicant",
        "source": f"applicant #{a['applicant_id']} — {a['name']}",
        "text": (
            f"applicant {a['name']}  Holder of {a['education']} "
            f"and he / she has   {a['applicant_experience']} years of experience. "
            f"and his/her : {a['skills']}. "
            f" Job applicant {a['title']} In {a['city']}."
        )
    })

print(" Number of doc :", len(docs))
print(docs[0]["text"])

 Number of doc : 38
Job: Frontend Developer  in company Tech Vision city Riyadh. salary: 12000 riyal.  requied skilles: Vue.js,JavaScript,HTML,CSS.  experience years: 2 years.


In [29]:
def chunk_text(text, size=400, overlap=80):
    if len(text) <= size:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

chunks = []
for d in docs:
    for i, part in enumerate(chunk_text(d["text"])):
        chunks.append({
            "chunk_id": f"{d['id']}_{len(chunks)}_c{i}",
            "doc_id": d["id"],
            "source": d["source"],
            "type": d["type"],
            "text": part
        })

print(f" Number of video : {len(chunks)}")
print(chunks[0])


 Number of video : 38
{'chunk_id': 'job_1_0_c0', 'doc_id': 'job_1', 'source': 'Job #1 — Frontend Developer', 'type': 'job', 'text': 'Job: Frontend Developer  in company Tech Vision city Riyadh. salary: 12000 riyal.  requied skilles: Vue.js,JavaScript,HTML,CSS.  experience years: 2 years.'}


In [30]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

texts = [c["text"] for c in chunks]
embeddings = embedder.encode(texts, show_progress_bar=True,
                             normalize_embeddings=True)

print(" Vector dimensions:", embeddings.shape)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

 Vector dimensions: (38, 384)


In [31]:
import chromadb

client = chromadb.Client()
try:
    client.delete_collection("recruitment")
except Exception:
    pass

collection = client.create_collection(
    name="recruitment", metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[c["chunk_id"] for c in chunks],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{"source": c["source"], "type": c["type"],
                "doc_id": c["doc_id"]} for c in chunks]
)

print(" Stoured , Number of Files :", collection.count())



 Stoured , Number of Files : 38


In [32]:
from rank_bm25 import BM25Okapi
import re

def tokenize(t):
    return re.findall(r"[\w\u0600-\u06FF]+", t.lower())

bm25 = BM25Okapi([tokenize(t) for t in texts])
print("index BM25 Ready — Number of documentes:", len(texts))


index BM25 Ready — Number of documentes: 38


In [33]:
import numpy as np

def dense_search(query, k=10):
    qv = embedder.encode([query], normalize_embeddings=True)
    res = collection.query(query_embeddings=qv.tolist(), n_results=k)
    return res["ids"][0]

def sparse_search(query, k=10):
    scores = bm25.get_scores(tokenize(query))
    top = np.argsort(scores)[::-1][:k]
    return [chunks[i]["chunk_id"] for i in top]

def rrf_fuse(dense_ids, sparse_ids, k=60):
    """Reciprocal Rank Fusion: score = Σ 1/(k + rank)"""
    fused = {}
    for rank, cid in enumerate(dense_ids, 1):
        fused[cid] = fused.get(cid, 0) + 1 / (k + rank)
    for rank, cid in enumerate(sparse_ids, 1):
        fused[cid] = fused.get(cid, 0) + 1 / (k + rank)
    return sorted(fused.items(), key=lambda x: x[1], reverse=True)

def hybrid_search(query, k=10):
    d, s = dense_search(query, k), sparse_search(query, k)
    return d, s, rrf_fuse(d, s)


q = "Who has experience in Python و Spark؟"
d, s, fused = hybrid_search(q)

print("Dense First 3 :", d[:3])
print("BM25  First 3 :", s[:3])
print("RRF   First 3 :", [c for c, _ in fused[:3]])




Dense First 3 : ['app_1006_25_c0', 'app_1006_26_c0', 'app_1006_27_c0']
BM25  First 3 : ['app_1007_32_c0', 'app_1007_31_c0', 'app_1007_30_c0']
RRF   First 3 : ['app_1006_28_c0', 'app_1008_34_c0', 'app_1008_36_c0']


In [34]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
chunk_map = {c["chunk_id"]: c for c in chunks}

def rerank(query, fused_ids, top_n=5):
    cands = [chunk_map[cid] for cid, _ in fused_ids[:10]]
    scores = reranker.predict([[query, c["text"]] for c in cands])
    ranked = sorted(zip(cands, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]

top = rerank(q, fused)
for c, sc in top:
    print(f"[{sc:.3f}] {c['source']}")


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[3.157] applicant #1006 — Updated A
[3.157] applicant #1006 — Updated A
[3.157] applicant #1006 — Updated A
[3.157] applicant #1006 — Updated A
[-1.388] applicant #1008 — Reem Saleh


In [35]:
def answer(query, top_n=4):
    d, s, fused = hybrid_search(query)
    ranked = rerank(query, fused, top_n)

    print("=" * 70)
    print("The Question:", query)
    print("=" * 70)
    print("\n Answer based on retrieved context: \n")

    for i, (c, sc) in enumerate(ranked, 1):
        print(f"[{i}] {c['text']}")
        print(f"    ↳ Source : {c['source']} | The Connection: {sc:.3f}\n")

    print("the reviewer:", ", ".join(f"[{i}] {c['source']}"
                                for i, (c, _) in enumerate(ranked, 1)))

answer(q)


The Question: Who has experience in Python و Spark؟

 Answer based on retrieved context: 

[1] applicant Updated A  Holder of Master and he / she has   9 years of experience. and his/her : Python, Spark, Kafka.  Job applicant Data Engineer In Jeddah.
    ↳ Source : applicant #1006 — Updated A | The Connection: 3.157

[2] applicant Updated A  Holder of Master and he / she has   9 years of experience. and his/her : Python, Spark, Kafka.  Job applicant Data Engineer In Jeddah.
    ↳ Source : applicant #1006 — Updated A | The Connection: 3.157

[3] applicant Updated A  Holder of Master and he / she has   9 years of experience. and his/her : Python, Spark, Kafka.  Job applicant Data Engineer In Jeddah.
    ↳ Source : applicant #1006 — Updated A | The Connection: 3.157

[4] applicant Updated A  Holder of Master and he / she has   9 years of experience. and his/her : Python, Spark, Kafka.  Job applicant Data Engineer In Jeddah.
    ↳ Source : applicant #1006 — Updated A | The Connection: 3.15

In [36]:
questions = [
"Who has experience with Python and Spark?",
    "What jobs are available in Riyadh?",
    "Which job offers the highest salary?",
    "Who are the suitable candidates for the Machine Learning Engineer position?",
    "Who has more than 4 years of experience?",
]

for question in questions:
    answer(question, top_n=3)
    print("\n")



The Question: Who has experience with Python and Spark?

 Answer based on retrieved context: 

[1] applicant Sara Khalid  Holder of Master and he / she has   4 years of experience. and his/her : Python, Spark, Kafka.  Job applicant Data Engineer In Jeddah.
    ↳ Source : applicant #1002 — Sara Khalid | The Connection: 7.350

[2] applicant Sara Khalid  Holder of Master and he / she has   4 years of experience. and his/her : Python, Spark, Kafka.  Job applicant Data Engineer In Jeddah.
    ↳ Source : applicant #1002 — Sara Khalid | The Connection: 7.350

[3] applicant Sara Khalid  Holder of Master and he / she has   4 years of experience. and his/her : Python, Spark, Kafka.  Job applicant Data Engineer In Jeddah.
    ↳ Source : applicant #1002 — Sara Khalid | The Connection: 7.350

the reviewer: [1] applicant #1002 — Sara Khalid, [2] applicant #1002 — Sara Khalid, [3] applicant #1002 — Sara Khalid


The Question: What jobs are available in Riyadh?

 Answer based on retrieved context: 

[

In [37]:
import json, os

os.makedirs(f"{BASE}/output", exist_ok=True)
log = []

for question in questions:
    d, s, fused = hybrid_search(question)
    ranked = rerank(question, fused, 3)
    log.append({
        "question": question,
        "dense_top3": d[:3],
        "bm25_top3": s[:3],
        "rrf_top3": [c for c, _ in fused[:3]],
        "final_after_rerank": [
            {"source": c["source"], "score": float(sc)} for c, sc in ranked]
    })

with open(f"{BASE}/output/rag_results.json", "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)

print("Saved")


Saved
